In [12]:
import math
import random
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

SEED = 7
DATA_DIR = Path('shakespeare-dataset-main/text')
BLOCK_SIZE, BATCH_SIZE, NUM_WORKERS = 64, 128, 0
DIM, NUM_LAYERS, NUM_HEADS, FF_DIM, DROPOUT = 128, 6, 4, 512, 0.1
DIFFUSION_STEPS, BETA_START, BETA_END = 1000, 1e-4, 0.02
LEARNING_RATE, WEIGHT_DECAY, GRAD_CLIP_NORM = 3e-4, 1e-2, 1.0
TRAIN_STEPS, LOG_EVERY, SAMPLE_EVERY = 10_000, 50, 1_000
ROUNDING_WEIGHT, COVERAGE_WEIGHT, CONTAINMENT_WEIGHT, PRIOR_SAMPLES = 1.0, 1.0, 1.0, 1_024
SAMPLE_BATCH_SIZE, SAMPLE_REVERSE_STEPS = 2, 250

device = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
random.seed(SEED)
print(f'Using {device}')


Using mps


In [13]:
def clean_folger_text(raw):
    lines = raw.replace('\r\n', '\n').replace('\r', '\n').splitlines(keepends=True)
    return ''.join(lines[7:]).lstrip('\n')

paths = sorted(DATA_DIR.glob('*.txt'))
if not paths:
    raise FileNotFoundError(f'No .txt files in {DATA_DIR}')
text = '\n\n'.join(clean_folger_text(p.read_text(encoding='utf-8')) for p in paths)
chars = sorted(set(text))
stoi = {char: index for index, char in enumerate(chars)}
itos = dict(enumerate(chars))
tokens = torch.tensor([stoi[c] for c in text], dtype=torch.long)
token_probabilities = torch.bincount(tokens, minlength=len(chars)).float()
token_probabilities /= token_probabilities.sum()

class CharacterBlocks(Dataset):
    def __init__(self, values, block_size):
        self.values = values
        self.block_size = block_size
    def __len__(self): return len(self.values) // self.block_size
    def __getitem__(self, index):
        start = index * self.block_size
        return self.values[start:start + self.block_size]

dataset = CharacterBlocks(tokens, BLOCK_SIZE)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS)
print(f'Loaded {len(paths)} works | {len(text):,} characters | vocab={len(chars)} | batch={tuple(next(iter(loader)).shape)}')


Loaded 42 works | 5,319,139 characters | vocab=84 | batch=(128, 64)


In [14]:
# Gaussian embedding
class GaussianEmbedding(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.mu = nn.Parameter(torch.randn(vocab_size, dim) * .02)
        initial_std = F.softplus(torch.randn(vocab_size, dim) * 0.25)

        # Initiatilize with standard aggregate variance
        target_std_rms = (1 - self.mu.detach().square().mean()).sqrt()
        initial_std = initial_std * target_std_rms / initial_std.square().mean().sqrt()
        self.raw_std = nn.Parameter(torch.log(torch.expm1(initial_std)))
    @property
    def std(self): return F.softplus(self.raw_std)
    def sample(self, ids):
        mean = self.mu[ids]
        return mean + self.std[ids] * torch.randn_like(mean)
    def squared_distances(self, vectors):
        return vectors.square().sum(-1, keepdim=True) + self.mu.square().sum(-1) - 2 * vectors @ self.mu.T

# Model 
class SimpleGaussianDiffusionLM(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = GaussianEmbedding(vocab_size, DIM)
        self.position, self.timestep = nn.Embedding(BLOCK_SIZE, DIM), nn.Embedding(DIFFUSION_STEPS + 1, DIM)
        layer = nn.TransformerEncoderLayer(DIM, NUM_HEADS, FF_DIM, DROPOUT, batch_first=True, norm_first=False, activation='gelu')
        self.transformer = nn.TransformerEncoder(layer, NUM_LAYERS)
        self.final_norm, self.x0_head = nn.LayerNorm(DIM), nn.Linear(DIM, DIM)
    def forward(self, x_t, timesteps):
        pos = torch.arange(x_t.size(1), device=x_t.device)
        hidden = x_t + self.position(pos).unsqueeze(0) + self.timestep(timesteps).unsqueeze(1)
        return self.x0_head(self.final_norm(self.transformer(hidden)))
    def rounding_logits(self, vectors): return -self.embedding.squared_distances(vectors)
    def round_to_tokens(self, vectors): return self.rounding_logits(vectors).argmax(-1)

model = SimpleGaussianDiffusionLM(len(chars)).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')


Parameters: 1,364,224


## Diffusion objective

Loss is denoising MSE plus character-rounding cross-entropy plus a sampled bidirectional KL between the empirical-frequency-weighted token-Gaussian mixture and the standard-normal prior. The two directions provide coverage and containment without constraining individual token variances.


In [15]:
betas = torch.linspace(BETA_START, BETA_END, DIFFUSION_STEPS, device=device)
alpha_bar = torch.cat((torch.ones(1, device=device), torch.cumprod(1 - betas, 0)))

def q_sample(x0, t):
    alpha = alpha_bar[t].view(-1, 1, 1)
    return alpha.sqrt() * x0 + (1 - alpha).sqrt() * torch.randn_like(x0)

@torch.no_grad()
def aggregate_statistics():
    probabilities = token_probabilities.to(device)
    means = model.embedding.mu
    variances = model.embedding.std.square()
    aggregate_mean = (probabilities[:, None] * means).sum(dim=0)
    aggregate_second_moment = (probabilities[:, None] * (means.square() + variances)).sum(dim=0)
    aggregate_variance = aggregate_second_moment - aggregate_mean.square()
    return aggregate_mean.mean(), aggregate_variance.mean()

def standard_log_prob(values):
    return -.5 * (values.square() + math.log(2 * math.pi)).sum(-1)

def mixture_log_prob(values):
    std = model.embedding.std
    scaled = (values[:, None] - model.embedding.mu) / std
    component_log_prob = -.5 * (scaled.square() + 2 * std.log() + math.log(2 * math.pi)).sum(-1)
    return torch.logsumexp(component_log_prob + token_probabilities.to(device).log(), dim=-1)

def aggregate_kl_loss():
    component_ids = torch.multinomial(token_probabilities.to(device), PRIOR_SAMPLES, replacement=True)
    from_mixture = model.embedding.sample(component_ids)
    from_prior = torch.randn_like(from_mixture)
    coverage = (standard_log_prob(from_prior) - mixture_log_prob(from_prior)).mean()
    containment = (mixture_log_prob(from_mixture) - standard_log_prob(from_mixture)).mean()
    return COVERAGE_WEIGHT * coverage + CONTAINMENT_WEIGHT * containment
def loss_terms(ids):
    x0 = model.embedding.sample(ids)
    t = torch.randint(1, DIFFUSION_STEPS + 1, (ids.size(0),), device=device)
    prediction = model(q_sample(x0, t), t)
    mse = F.mse_loss(prediction, x0)
    rounding = F.cross_entropy(model.rounding_logits(x0).flatten(0, 1), ids.flatten())
    weighted_prior = aggregate_kl_loss()
    weighted_ce = ROUNDING_WEIGHT * rounding
    total = mse + weighted_ce + weighted_prior
    return total, mse, weighted_ce, weighted_prior
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


In [16]:
def step(batch):
    optimizer.zero_grad(set_to_none=True)
    values = loss_terms(batch.to(device))
    values[0].backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
    optimizer.step()
    aggregate_mean, aggregate_variance = aggregate_statistics()
    return *values, aggregate_mean, aggregate_variance, grad_norm
def train(steps=TRAIN_STEPS):
    model.train()
    iterator = iter(loader)
    running = torch.zeros(7)
    for number in range(1, steps + 1):
        try: batch = next(iterator)
        except StopIteration:
            iterator = iter(loader)
            batch = next(iterator)
        values = step(batch)
        running += torch.tensor([value.item() for value in values])
        if number % LOG_EVERY == 0 or number == steps:
            mean = running / min(LOG_EVERY, number)
            print(
                f'step {number}/{steps} | total={mean[0]:.4f} | '
                f'mse={mean[1]:.4f} | weighted_ce={mean[2]:.4f} | weighted_prior={mean[3]:.6f} | '
                f'aggregate_mean={mean[4]:.4f} | aggregate_variance={mean[5]:.4f} | '
                f'grad_norm={mean[6]:.4f}'
            )
            running.zero_()
        if number % SAMPLE_EVERY == 0:
            print(f'\nSamples at step {number}:')
            for token_block in sample().cpu():
                print(''.join(itos[token.item()] for token in token_block))
                print()


In [18]:
@torch.no_grad()
def sample(batch_size=SAMPLE_BATCH_SIZE, reverse_steps=SAMPLE_REVERSE_STEPS):
    was_training = model.training
    model.eval()
    x_t = torch.randn(batch_size, BLOCK_SIZE, DIM, device=device)
    times = torch.linspace(DIFFUSION_STEPS, 0, reverse_steps + 1, device=device).long()
    for current, next_time in zip(times[:-1], times[1:]):
        t = torch.full((batch_size,), current, device=device, dtype=torch.long)
        x0_hat = model(x_t, t)
        a_t, a_next = alpha_bar[current], alpha_bar[next_time]
        eps_hat = (x_t - a_t.sqrt() * x0_hat) / (1 - a_t).sqrt().clamp_min(1e-8)
        x_t = a_next.sqrt() * x0_hat + (1 - a_next).sqrt() * eps_hat
    result = model.round_to_tokens(x_t)
    model.train(was_training)
    return result

train()

step 50/10000 | total=4.1114 | mse=0.7828 | weighted_ce=2.8397 | weighted_prior=0.488962 | aggregate_mean=0.0013 | aggregate_variance=0.9935 | grad_norm=0.4447
step 100/10000 | total=4.0957 | mse=0.7871 | weighted_ce=2.8119 | weighted_prior=0.496705 | aggregate_mean=0.0013 | aggregate_variance=0.9934 | grad_norm=0.4416
step 150/10000 | total=4.0715 | mse=0.7848 | weighted_ce=2.7792 | weighted_prior=0.507553 | aggregate_mean=0.0012 | aggregate_variance=0.9934 | grad_norm=0.4423
step 200/10000 | total=4.0394 | mse=0.7772 | weighted_ce=2.7418 | weighted_prior=0.520407 | aggregate_mean=0.0012 | aggregate_variance=0.9934 | grad_norm=0.4425
step 250/10000 | total=4.0050 | mse=0.7773 | weighted_ce=2.7106 | weighted_prior=0.517155 | aggregate_mean=0.0012 | aggregate_variance=0.9932 | grad_norm=0.4433
step 300/10000 | total=3.9884 | mse=0.7745 | weighted_ce=2.6909 | weighted_prior=0.523014 | aggregate_mean=0.0011 | aggregate_variance=0.9932 | grad_norm=0.4426
step 350/10000 | total=3.9586 | mse

KeyboardInterrupt: 

In [ ]:
for token_block in sample(reverse_steps=250).cpu():
                print(''.join(itos[token.item()] for token in token_block))
                print()

 s
CsIs=mABcu SsOoli
irdY yficvhkrLtw
i nokor,EgSdnwr ohkeOC lwr

Ftiuydtsns o]su
kldvnp 
a s lrsU,tpv ohG k "btEctuaEtp rv
snsfhl

